# Section 3: Robustness, Where You Go Beyond (28 pts)

ECON 5200 Midterm Project · Replication Study

**AI is allowed in this notebook**, as a documented co-pilot under P.R.I.M.E. File every exchange, in full, in [`ai-appendix.md`](../ai-appendix.md) under the exercise it belongs to. If you cannot explain a line, you cannot submit it.

At least four exercises the original did not run. Choose from modern estimators, a specification curve, sensitivity to unobserved confounding, inference (clustering at the level of treatment assignment, a wild cluster bootstrap if clusters are few), bandwidth or instrument strength, and placebo tests, or justify an alternative. For each one: what you did, what you found, and what it implies about the original claim.

## Setup

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

# The repository root, whether this runs from notebooks/ (Jupyter) or from the root (reproduce.py).
ROOT = Path.cwd() if (Path.cwd() / "download_data.py").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
(ROOT / "output").mkdir(exist_ok=True)

from download_data import MY_PAPER, PAPERS, fetch

## Data

In [2]:
# Set MY_PAPER once, in download_data.py. fetch() downloads into data/ only if a file is missing.
if MY_PAPER:
    paths = fetch(MY_PAPER)
    print(PAPERS[MY_PAPER]["paper"])
    df = pd.read_stata(paths[0])
    print("Dataset shape:", df.shape)
    print(df[["lagdemvoteshare", "lagdemocrat", "demvoteshare"]].head())

else:
    print("TODO: set MY_PAPER in download_data.py (castle, card, lee, dw, hansen or abortion).")

  data/lmb-data.dta  7.2 MB
Lee (2008), incumbency advantage in US House elections
Dataset shape: (13588, 178)
   lagdemvoteshare  lagdemocrat  demvoteshare
0         0.469256          0.0      0.553026
1         0.469256          0.0      0.553026
2         0.553026          1.0      0.582441
3         0.553026          1.0      0.582441
4         0.539680          1.0      0.569626


## The four exercises

### Exercise 1: TODO name it

**What I did. I tested four different bandwidths (0.03, 0.05, 0.07, and 0.10) around the 50% cutoff using a local linear regression.

In [3]:

# Check variables and sample structure
print("Relevant columns:")
print([c for c in df.columns if c in [
    "state", "district", "year",
    "lagdemvoteshare", "lagdemocrat", "demvoteshare"
]])

print("\nComplete observations:",
      df[["lagdemvoteshare", "lagdemocrat", "demvoteshare"]]
      .dropna().shape[0])

print("\nRepeated election keys:",
      df.duplicated(["state", "district", "year"]).sum())


Relevant columns:
['state', 'district', 'year', 'demvoteshare', 'lagdemvoteshare', 'lagdemocrat']

Complete observations: 13566

Repeated election keys: 6446


In [4]:
# Exercise 1: Bandwidth sensitivity
# Keep one observation per election, consistent with Section 2
sample = df.dropna(
    subset=["lagdemvoteshare", "lagdemocrat", "demvoteshare"]
).drop_duplicates(subset=["state", "district", "year"]).copy()

sample["margin"] = sample["lagdemvoteshare"] - 0.5

results = []

for bw in [0.03, 0.05, 0.07, 0.10]:
    d = sample[sample["margin"].abs() <= bw].copy()

    model = smf.ols(
        "demvoteshare ~ lagdemocrat + margin + lagdemocrat:margin",
        data=d
    ).fit(cov_type="HC1")

    results.append({
        "Bandwidth": bw,
        "N": int(model.nobs),
        "Estimate": round(model.params["lagdemocrat"], 4),
        "SE": round(model.bse["lagdemocrat"], 4)
    })

bw_results = pd.DataFrame(results)
print(bw_results.to_string(index=False))


 Bandwidth    N  Estimate     SE
      0.03  768    0.0860 0.0135
      0.05 1286    0.0856 0.0104
      0.07 1807    0.0868 0.0089
      0.10 2509    0.0937 0.0077



**What I found.** The estimates range from 0.0856 to 0.0937, and all are positive. The results do not change much across the four bandwidths.

**What it implies about the original claim.** The positive effect remains across different bandwidths, which supports the paper's main finding. However, the estimate becomes slightly larger with a wider bandwidth.


### Exercise 2: TODO name it

**What I did. I tested fake cutoffs at 40% and 60% to check whether similar jumps appear away from the actual 50% threshold.

In [5]:
# Exercise 2: Placebo cutoffs
placebo_results = []

for cutoff in [0.40, 0.60]:
    d = sample[
        (sample["lagdemvoteshare"] - cutoff).abs() <= 0.05
    ].copy()

    # Keep observations on the same side of the actual cutoff
    d = d[
        (d["lagdemvoteshare"] < 0.5)
        if cutoff < 0.5
        else (d["lagdemvoteshare"] > 0.5)
    ].copy()

    d["fake_margin"] = d["lagdemvoteshare"] - cutoff
    d["fake_win"] = (d["fake_margin"] >= 0).astype(int)

    model = smf.ols(
        "demvoteshare ~ fake_win + fake_margin + fake_win:fake_margin",
        data=d
    ).fit(cov_type="HC1")

    placebo_results.append({
        "Cutoff": cutoff,
        "N": int(model.nobs),
        "Estimate": round(model.params["fake_win"], 4),
        "SE": round(model.bse["fake_win"], 4),
        "P-value": round(model.pvalues["fake_win"], 4)
    })

placebo_table = pd.DataFrame(placebo_results)
print(placebo_table.to_string(index=False))

 Cutoff    N  Estimate     SE  P-value
    0.4 1311   -0.0044 0.0107   0.6838
    0.6 1081    0.0045 0.0147   0.7595


**What I found.** The estimates are -0.0044 at 40% and 0.0045 at 60%. Both p-values are above 0.05, so neither result is statistically significant.

**What it implies about the original claim.** There is no clear jump at either fake cutoff. This supports the idea that the effect at 50% is related to actually winning the election, rather than a general pattern in the data.

### Exercise 3: TODO name it

**What I did.I compared linear and quadratic regression models around the 50% cutoff to see whether the results depend on the model specification.

In [6]:
# Exercise 3: Polynomial order
d = sample[sample["margin"].abs() <= 0.10].copy()
d["margin_sq"] = d["margin"] ** 2

models = {
    "Linear": "demvoteshare ~ lagdemocrat + margin + lagdemocrat:margin",
    "Quadratic": (
        "demvoteshare ~ lagdemocrat + margin + margin_sq "
        "+ lagdemocrat:margin + lagdemocrat:margin_sq"
    )
}

poly_results = []

for name, formula in models.items():
    model = smf.ols(formula, data=d).fit(cov_type="HC1")
    poly_results.append({
        "Model": name,
        "N": int(model.nobs),
        "Estimate": round(model.params["lagdemocrat"], 4),
        "SE": round(model.bse["lagdemocrat"], 4),
        "P-value": round(model.pvalues["lagdemocrat"], 4)
    })

poly_table = pd.DataFrame(poly_results)
print(poly_table.to_string(index=False))


    Model    N  Estimate     SE  P-value
   Linear 2509    0.0937 0.0077      0.0
Quadratic 2509    0.0854 0.0111      0.0


**What I found.** The linear model gives an estimate of 0.0937, while the quadratic model gives 0.0854. Both are statistically significant.

**What it implies about the original claim.** The result stays positive even when I change the model specification. This suggests that the main finding is not very sensitive to using a linear or quadratic model.

### Exercise 4: TODO name it

**What I did. I compared heteroskedasticity-robust standard errors with state-clustered standard errors to see whether the statistical significance changes.

In [7]:
# Exercise 4: Alternative inference
d = sample[sample["margin"].abs() <= 0.05].copy()

formula = (
    "demvoteshare ~ lagdemocrat + margin + lagdemocrat:margin"
)

hc1_model = smf.ols(formula, data=d).fit(cov_type="HC1")

cluster_model = smf.ols(formula, data=d).fit(
    cov_type="cluster",
    cov_kwds={"groups": d["state"]}
)

inference_table = pd.DataFrame([
    {
        "Method": "HC1",
        "N": int(hc1_model.nobs),
        "Estimate": round(hc1_model.params["lagdemocrat"], 4),
        "SE": round(hc1_model.bse["lagdemocrat"], 4),
        "P-value": round(hc1_model.pvalues["lagdemocrat"], 4)
    },
    {
        "Method": "State clustered",
        "N": int(cluster_model.nobs),
        "Estimate": round(cluster_model.params["lagdemocrat"], 4),
        "SE": round(cluster_model.bse["lagdemocrat"], 4),
        "P-value": round(cluster_model.pvalues["lagdemocrat"], 4)
    }
])

print(inference_table.to_string(index=False))


         Method    N  Estimate     SE  P-value
            HC1 1286    0.0856 0.0104      0.0
State clustered 1286    0.0856 0.0138      0.0


**What I found.** The estimate stays at 0.0856 with both methods. The standard error increases from 0.0104 to 0.0138 when I cluster by state, but the result is still statistically significant.

**What it implies about the original claim.** The main finding remains significant even after allowing observations within the same state to be correlated. This makes the result more convincing, although the uncertainty is slightly larger.
